# Indicadores de comportamiento exportador por subpartida homologada

Este cuaderno calcula ocho indicadores de comportamiento exportador para el periodo 2011–2025. La tabla resultante contiene una fila por subpartida homologada. También se calcula la matriz de correlaciones de Spearman entre los ocho indicadores.

## Archivo de entrada

`datos/procesados/exportaciones_1a1_sin_agregacion.parquet`, generado por el cuaderno 02.

## Archivos de salida

- `datos/procesados/indicadores_subpartidas_2011_2025.parquet`, tabla de indicadores que utiliza el cuaderno 04 para el clustering.
- `datos/procesados/indicadores_subpartidas_2011_2025.csv`, copia de la misma tabla para consulta.
- En `resultados/indicadores/` se guardan la matriz de Spearman, el mapa de calor y la tabla de correlaciones entre pares de indicadores.

In [ ]:
# Se importan las dependencias necesarias
from pathlib import Path
import numpy as np
import pandas as pd

## 1. Parámetros

In [ ]:
# 1. Configuración de rutas de entrada y salida.
# Por defecto, se utiliza la carpeta donde se encuentra el cuaderno.
carpeta_entrada = Path("datos/procesados")
carpeta_salida = Path("resultados/indicadores")
archivo_base = "exportaciones_1a1_sin_agregacion.parquet"

# Correspondencia entre las variables del análisis y las columnas de la base.
col_subpartida = "subpartida_homologada"
col_anio = "Año"
col_mes = "Mes"
col_destino = "Destino/Origen"
col_fob = "US$ millones"
col_tm = "TM"

# Periodo de análisis.
anio_inicio = 2011
anio_fin = 2025

## 2. Lectura de la base

In [ ]:
# 2. Lectura de la base y validación de los datos de entrada.
def leer(ruta):
    """Lee un archivo parquet o csv según su extensión."""
    if ruta.suffix == ".parquet":
        return pd.read_parquet(ruta)
    if ruta.suffix == ".csv":
        return pd.read_csv(ruta, dtype={col_subpartida: str})
    raise ValueError(f"Formato no reconocido: {ruta.suffix}")

ruta_base = carpeta_entrada / archivo_base
if not ruta_base.exists():
    raise FileNotFoundError(
        f"No se encontró la base homologada definitiva: {ruta_base}. "
        "Revisar carpeta_entrada y archivo_base en la celda de parámetros."
    )
base = leer(ruta_base)

# Verificación de la versión definitiva de la base homologada.
assert len(base) == 1_671_785, (
    f"La base tiene {len(base):,} filas; se esperan 1 671 785. "
    "Comprobar que el archivo sea exportaciones_1a1_sin_agregacion.parquet."
)
assert base[col_subpartida].nunique() == 5_626, (
    "La base no contiene las 5 626 subpartidas homologadas esperadas."
)
assert np.isclose(base[col_fob].sum(), 706_350.99033953, atol=1e-5, rtol=0), (
    "El valor FOB no coincide con la selección definitiva."
)

# Selección de las columnas necesarias para el cálculo de indicadores.
base = base[[col_subpartida, col_anio, col_mes, col_destino, col_fob, col_tm]].copy()

# Definición del periodo de análisis.
anios = list(range(anio_inicio, anio_fin + 1))
n_anios = len(anios)

# Resumen de la base utilizada en el análisis.
print("Registros:", len(base))
print("Subpartidas:", base[col_subpartida].nunique())
print("Años:", base[col_anio].min(), "a", base[col_anio].max())

## 3. Actividad exportadora
Un año (o un mes) se considera activo cuando existe al menos un registro con FOB positivo o TM positiva.

In [ ]:
# 3. Construcción de las matrices anuales de FOB y actividad comercial.
base["activo"] = (base[col_fob] > 0) | (base[col_tm] > 0)

# FOB acumulado por subpartida y año; los años sin registros se completan con cero.
fob_anual = (base.pivot_table(index=col_subpartida, columns=col_anio, values=col_fob,
                              aggfunc="sum", fill_value=0)
                 .reindex(columns=anios, fill_value=0))

# Indicador de actividad anual: 1 si existe actividad comercial y 0 en caso contrario.
activo_anual = (base.groupby([col_subpartida, col_anio])["activo"].any()
                    .unstack(fill_value=False)
                    .reindex(index=fob_anual.index, columns=anios, fill_value=False)
                    .astype(int))

print("Tabla de FOB anual:", fob_anual.shape)

## 4. Indicadores

In [ ]:
# 4. Cálculo de los ocho indicadores candidatos por subpartida homologada.
ind = pd.DataFrame(index=fob_anual.index)

# 4.1. Magnitud económica: logaritmo del FOB total (millones de US$)
ind["fob_total_mill"] = fob_anual.sum(axis=1)
ind["log_fob_total"] = np.log1p(ind["fob_total_mill"])

# 4.2. Continuidad: número de años activos
ind["anios_activos"] = activo_anual.sum(axis=1)

# 4.3. Intermitencia: número de episodios de actividad exportadora
# Se cuenta un episodio cuando un año activo está precedido por uno inactivo.
# Para el primer año del periodo, se considera que el año anterior fue inactivo.
previo = activo_anual.shift(1, axis=1, fill_value=0)
ind["episodios_exportadores"] = ((activo_anual == 1) & (previo == 0)).sum(axis=1)

# 4.4. Regularidad mensual: meses activos / (12 x años activos)
meses_activos = (base[base["activo"]]
                 .drop_duplicates([col_subpartida, col_anio, col_mes])
                 .groupby(col_subpartida).size()
                 .reindex(ind.index, fill_value=0))
ind["meses_activos"] = meses_activos
ind["regularidad_mensual"] = meses_activos / (12 * ind["anios_activos"])

# 4.5 y 4.6. Se incluyen los registros activos con un valor en la columna Destino/Origen.
con_destino = base[base["activo"] & base[col_destino].notna()]

# 4.5. HHI de destinos: suma de los cuadrados de la participación de cada destino en el FOB
fob_destino = (con_destino[con_destino[col_fob] > 0]
               .groupby([col_subpartida, col_destino])[col_fob].sum())
participacion = fob_destino / fob_destino.groupby(level=0).transform("sum")
ind["hhi_destinos_periodo"] = (participacion ** 2).groupby(level=0).sum()

# 4.6. Mercados distintos: número de destinos diferentes con actividad
ind["mercados_distintos_periodo"] = (con_destino.groupby(col_subpartida)[col_destino]
                                     .nunique()
                                     .reindex(ind.index, fill_value=0))

# 4.7. Recencia: años transcurridos desde la última actividad hasta el final del periodo
ultimo_activo = (activo_anual * np.array(anios)).max(axis=1)
ind["anios_desde_ultimo_activo"] = np.where(ind["anios_activos"] > 0,
                                            anio_fin - ultimo_activo, np.nan)

# 4.8. Pendiente logarítmica del FOB anual (regresión de ln(1 + FOB) sobre el año). Se utilizan los 15 años del periodo de estudio.
y = np.log1p(fob_anual.to_numpy())
x = np.array(anios, dtype=float)
x_centrado = x - x.mean()
ind["pendiente_log_fob_anual"] = (y * x_centrado).sum(axis=1) / (x_centrado ** 2).sum()

ind = ind.reset_index().rename(columns={col_subpartida: "subpartida_homologada"})
print(ind.shape)

## 5. Revisión
Se comprueba que no haya faltantes inesperados y que los valores estén dentro de lo posible.

In [ ]:
# 5. Validación y revisión descriptiva de los indicadores candidatos.
columnas = ["log_fob_total", "anios_activos", "episodios_exportadores", "regularidad_mensual",
            "hhi_destinos_periodo", "mercados_distintos_periodo",
            "anios_desde_ultimo_activo", "pendiente_log_fob_anual"]

# Verificación de la cantidad de subpartidas y de la integridad de los indicadores.
assert len(ind) == 5_626
assert not ind[columnas].isna().any().any(), "Hay indicadores faltantes."
assert np.isfinite(ind[columnas].to_numpy(dtype=float)).all(), "Hay valores infinitos."

# Resumen estadístico de los indicadores calculados.
display(ind[columnas].describe().round(3).T)

# Comprobación de los rangos y relaciones esperados entre indicadores.
assert ind["anios_activos"].between(0, n_anios).all()
assert (ind["episodios_exportadores"] <= ind["anios_activos"]).all()
assert ind["regularidad_mensual"].dropna().between(0, 1).all()
assert ind["hhi_destinos_periodo"].dropna().between(0, 1).all()

## 6. Guardado

In [ ]:
# Tabla de indicadores: una fila por subpartida homologada
ruta_indicadores = carpeta_entrada / "indicadores_subpartidas_2011_2025.parquet"
ind.to_parquet(ruta_indicadores, index=False)

# Copia en CSV para consulta
ind.to_csv(ruta_indicadores.with_suffix(".csv"), index=False, encoding="utf-8-sig")

# Carpeta para la matriz y la figura de correlaciones de Spearman
carpeta_salida.mkdir(parents=True, exist_ok=True)

## 7. Alcance del cuaderno

Se construyen ocho indicadores y realiza el análisis de sus correlaciones mediante el coeficiente de Spearman.
Adicionalmente, se generan dos columnas auxiliares (`fob_total_mill` y `meses_activos`) que intervienen en el cálculo de los indicadores, pero no forman parte del conjunto de ocho variables candidatas.

In [ ]:
# 7. Análisis de correlaciones de Spearman.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# 7.1 Selección de los ocho indicadores candidatos.
variables = columnas

datos = ind[variables].copy()

# 7.2 Cálculo de la matriz de correlaciones de Spearman.
correlacion = datos.corr(method="spearman")

print("\nMATRIZ DE CORRELACIONES DE SPEARMAN")
display(correlacion.round(3))

# 7.3 Representación de las correlaciones mediante un mapa de calor.
fig, ax = plt.subplots(figsize=(11, 9))

im = ax.imshow(
    correlacion.values,
    cmap="coolwarm",
    vmin=-1,
    vmax=1
)

ax.set_xticks(range(len(variables)))
ax.set_yticks(range(len(variables)))

ax.set_xticklabels(variables, rotation=45, ha="right")
ax.set_yticklabels(variables)

for i in range(len(variables)):
    for j in range(len(variables)):
        valor = correlacion.iloc[i, j]
        color = "white" if abs(valor) >= 0.65 else "black"
        ax.text(
            j, i, f"{valor:.2f}",
            ha="center", va="center",
            color=color, fontsize=9
        )

fig.colorbar(im, ax=ax, label="Rho de Spearman")
ax.set_title("Correlación de Spearman - Indicadores exportadores")
plt.tight_layout()

plt.savefig(
    carpeta_salida / "matriz_correlacion_spearman.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# 7.4 Construcción y ordenamiento de las correlaciones entre pares de indicadores.
pares = []

for i in range(len(variables)):
    for j in range(i + 1, len(variables)):
        rho = correlacion.iloc[i, j]

        pares.append({
            "Variable 1": variables[i],
            "Variable 2": variables[j],
            "Spearman": rho,
            "Valor absoluto": abs(rho)
        })

tabla_pares = pd.DataFrame(pares).sort_values(
    "Valor absoluto", ascending=False
)

print("\nTODAS LAS CORRELACIONES ORDENADAS")
display(tabla_pares.round(3))

# 7.5 Exportación de las matrices y tablas de correlación.
correlacion.to_csv(
    carpeta_salida / "matriz_correlacion_spearman.csv",
    encoding="utf-8-sig"
)

tabla_pares.to_csv(
    carpeta_salida / "correlaciones_ordenadas.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nAnálisis terminado.")
print("Resultados guardados en:", carpeta_salida)
